In [30]:
# -----------------------------
# Full Notebook: Impulse Response from Differential Equation
# -----------------------------
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

# -----------------------------
# Symbols and functions
# -----------------------------
t = sp.symbols('t', real=True)
alpha = sp.symbols('alpha', positive=True)

y = sp.Function('y')(t)
x = sp.Function('x')(t)

# -----------------------------
# System differential equation
# -----------------------------
ode = sp.Eq(sp.diff(y,t) + alpha*y, sp.diff(x,t) - 2*x)

print("System differential equation:")
sp.pprint(ode)

# -----------------------------
# Impulse input
# -----------------------------
print("\nImpulse input:")
print("x(t) = δ(t)")

print("\nThen the system becomes:")
print("d/dt h(t) + α h(t) = δ'(t) - 2δ(t)")

# -----------------------------
# Auxiliary impulse response
# -----------------------------
print("\nAuxiliary system:")
print("d/dt h_beta(t) + α h_beta(t) = δ(t)")

# symbols
t, alpha = sp.symbols('t alpha', positive=True, real=True)
h_beta_func = sp.Function('h_beta')(t)

# auxiliary ODE
ode_aux = sp.Eq(sp.Derivative(h_beta_func, t) + alpha*h_beta_func, sp.DiracDelta(t))

# solve symbolically with initial condition h_beta(0+) = 1
h_beta_sol = sp.dsolve(ode_aux, h_beta_func, ics={h_beta_func.subs(t,0):1})

print("\nSolution of auxiliary system:")
sp.pprint(h_beta_sol.rhs)

# -----------------------------
# Symbolic derivation of impulse response
# -----------------------------
dh_beta = sp.diff(h_beta_sol.rhs, t)
h_symbolic = sp.simplify(dh_beta - 2*h_beta_sol.rhs)

print("\nContinuous component of impulse response h(t):")
sp.pprint(h_symbolic)

print("\nNumeric implementation:")
print("h_c(t) = -(alpha + 2) * exp(-alpha*t) * u(t)")

# -----------------------------
# Numeric implementation
# -----------------------------
def h_c_numeric(t_vals, a):
    u = np.heaviside(t_vals,1)
    return -(a+2)*np.exp(-a*t_vals)*u

# -----------------------------
# Plot function with slider
# -----------------------------
def plot_impulse(alpha_val):
    t_vals = np.linspace(-1,5,1000)

    h_sym_func = sp.lambdify((t,alpha), h_symbolic, 'numpy')
    h_sym_vals = h_sym_func(t_vals, alpha_val) * np.heaviside(t_vals,1)
    h_num_vals = h_c_numeric(t_vals, alpha_val)

    plt.figure(figsize=(8,5))
    plt.plot(t_vals,h_sym_vals,label="Symbolic")
    plt.plot(t_vals,h_num_vals,'--',label="Numeric")

    # delta arrow
    plt.annotate('',
                 xy=(0,1),
                 xytext=(0,0),
                 arrowprops=dict(width=2, headwidth=8, color='red'))
    # delta label
    plt.text(0.15, 1, r'$\delta(t)$', color='red', fontsize=14, va='center')

    plt.axhline(0)
    plt.axvline(0)

    # y-limits ώστε να φαίνεται το delta arrow
    plt.ylim(bottom=min(h_sym_vals)-1, top=2)

    plt.title("Impulse Response h(t) continuous component")
    plt.xlabel("t")
    plt.ylabel("h_c(t)")

    plt.legend(loc="lower right")
    plt.show()

# -----------------------------
# Slider
# -----------------------------
interact(plot_impulse,
         alpha_val=FloatSlider(value=3,min=0.5,max=10,step=0.5))

System differential equation:
         d                    d       
α⋅y(t) + ──(y(t)) = -2⋅x(t) + ──(x(t))
         dt                   dt      

Impulse input:
x(t) = δ(t)

Then the system becomes:
d/dt h(t) + α h(t) = δ'(t) - 2δ(t)

Auxiliary system:
d/dt h_beta(t) + α h_beta(t) = δ(t)

Solution of auxiliary system:
 -α⋅t
ℯ    

Continuous component of impulse response h(t):
          -α⋅t
(-α - 2)⋅ℯ    

Numeric implementation:
h_c(t) = -(alpha + 2) * exp(-alpha*t) * u(t)


interactive(children=(FloatSlider(value=3.0, description='alpha_val', max=10.0, min=0.5, step=0.5), Output()),…

<function __main__.plot_impulse(alpha_val)>